#### 質問：
a. 測定結果を古典側で処理して、次ステップの回転角を決める閉ループを、1ジョブの中の中間測定＋条件付きゲートで書けますか。それともステップごとにジョブを分ける必要がありますか。  
b. その場合、HQCの固定項5はジョブごとですか、ステップごとですか。

#### 回答:
a. 条件によって、回転角度がすでに決まっているのであれば、可能です。下記の例を参照してください。  
動的に回転角度を変えたい場合にはGuppyを使う必要があります。  
Guppyを利用して、Runtime時に生成された乱数を使ってangleを決める方法  
https://docs.quantinuum.com/systems/trainings/helios/getting_started/rng.html#use-case-2-generate-random-angles  
が参考になるかと思います。  
b. 上で解決するのであれば、量子回路のHQCはシンタックスチェッカーで計算可能です。

In [1]:
from pytket import Circuit
from pytket.circuit import Qubit, Bit
from pytket.circuit.display import render_circuit_jupyter

In [2]:
def measure_depend_theta_circuit(max_step: int = 5) -> Circuit:
    """
    Pytket version of the Guppy measure_depend_theta_circuit example.

    The dynamic loop is unrolled to `max_attempts`; each attempt uses fresh ancillas
    and records their measurement outcomes for classical post-processing.
    """
    circ = Circuit()

    q = Qubit("q", 0)
    circ.add_qubit(q)

    q_out = Bit("q_out", 0)
    circ.add_bit(q_out)
    theta = 1

    measurements = [
        Bit("a_m", i)
        for i in range(max_step)
    ]
    
    circ.H(q)
    for bit in measurements:
        circ.add_bit(bit)

    for i, a_m in enumerate(measurements):

    # 過去の測定値が 1 だった回数だけ Rx(0.5*(i+1)) を追加, 過去の測定値が 0 だった回数だけ Rx(-0.5*(i+1)) を追加
        circ.Rx(0.5*(i+1), q, condition_bits=[measurements[i]], condition_value=1)
        circ.Rx(-0.5*(i+1), q, condition_bits=[measurements[i]], condition_value=0)

        circ.Measure(q, a_m)

    circ.Measure(q, q_out)
    return circ

circ = measure_depend_theta_circuit(max_step=5)
render_circuit_jupyter(circ)

### AerBackend（シミュレータ）で検証

In [3]:
!pip install pytket-qiskit

In [4]:
from pytket.extensions.qiskit import AerBackend

In [5]:
b =AerBackend()

In [6]:
c = b.get_compiled_circuit(circ)

In [7]:
r = b.process_circuit(c, n_shots =100)

In [8]:
count = b.get_result(r)

In [18]:
count.get_counts([c.bits[4]])

Counter({(1,): 54, (0,): 46})

### H2-Emulator（qnexus）で検証

In [19]:
import qnexus as qnx
from pytket import Circuit
from datetime import datetime
import uuid

unique_suffix = uuid.uuid1()

project = qnx.projects.get_or_create("H2-Samples")
qnx.context.set_active_project(project)

device_name = "H2-Emulator"
config = qnx.QuantinuumConfig(device_name=device_name)

In [20]:
my_circuit = qnx.circuits.upload(
    circuit=circ, 
    name=f"repeat-until-success-{unique_suffix}"
)

In [23]:
result_ref = qnx.start_execute_job(
    programs=[my_circuit],
    n_shots=[100],
    backend_config=config,
    name=f"H2-Emulator-{unique_suffix}",
)

In [24]:
my_job_name_prefix = datetime.now()
my_comp = qnx.start_compile_job(
    name = f"my_circ compilation {my_job_name_prefix}",
    programs=[my_circuit],
    optimisation_level=3,
    backend_config = config
    )

In [25]:
#qnx.jobs.status(my_comp)
qnx.jobs.wait_for(my_comp)

JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 9, 18, 6, 31, 6, 817217, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 9, 18, 6, 31, 2, 608601, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 9, 18, 6, 30, 59, 878074, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 9, 18, 6, 31, 3, 145144, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)

In [26]:
compiled_circuits = [item.get_output() for item in qnx.jobs.results(my_comp)]
my_exe = qnx.start_execute_job(
    programs=compiled_circuits,
    name=f"my_circ execution {my_job_name_prefix}",
    n_shots=[100] * len(compiled_circuits),
    backend_config=config
)

In [28]:
#qnx.jobs.status(my_exe)
qnx.jobs.wait_for(my_exe)

JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 9, 18, 6, 32, 15, 233085, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 9, 18, 6, 31, 35, 494110, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 9, 18, 6, 31, 10, 259758, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 9, 18, 6, 32, 4, 795876, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)

In [29]:
execute_job_result_refs = qnx.jobs.results(my_exe)
execute_job_result_refs.df()
#実行結果
result = execute_job_result_refs[0].download_result()
result.get_counts([circ.bits[4]])

Counter({(1,): 71, (0,): 29})